# Speculative Decoding

# Helper Functions

In [14]:
import torch
from gpt_model import GPTModel

In [10]:
# detect what hardware is available
# basically, detect the best fastest available processor
# on our computer so that your GPT model doesn't run at 
# a snail's pace on the CPU if a better option exists.

# check for an NVIDIA graphics card with CUDA drivers installed
# GOLD STANDARD

def pick_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    # check for apple GPUs built into M1, M2 or M3 chips
    elif torch.backends.mps.is_available():
        # Only use GPU on a mac if the PyTorch supports it
        major, minor = map(int, torch.__version__.split(".")[:2])
        if (major, minor) >= (2, 9):
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    # fallback to CPU
    else:
        device = torch.device("cpu")

    return device

In [11]:
# Helper to download open weights for GPT-2.0, 
# already in PyTorch format.

import requests
import os

def download_weights_from_openai(filename):
    url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{filename}"
    
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        # 'verify=False' bypasses the SSL check entirely
        response = requests.get(url, verify=False)
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Successfully downloaded to {filename}!")
    else:
        print(f"{filename} already exists.")

In [19]:
# map the downloaded weights to my own class for GPT-2.0


def load_model(weights_path, config, device):
    model = GPTModel(config)

    # Load the state dictionary
    pretrained_state_dict = torch.load(
        weights_path, map_location=device, weights_only=True
    )

    # Create a new dict with updated keys
    new_state_dict = {}
    for key, value in pretrained_state_dict.items():
        new_key = key
        # Map attributes: 'trf_blocks' -> 'transformer_blocks',
        # 'att' -> 'mha', 'norm1' -> 'layerNorm1', 'norm2' -> 'layerNorm2'
        new_key = new_key.replace("tok_emb", "sem_emb")
        new_key = new_key.replace("trf_blocks", "transformer_blocks")
        new_key = new_key.replace(".att.", ".mha.")
        new_key = new_key.replace(".norm1.", ".layerNorm1.")
        new_key = new_key.replace(".norm2.", ".layerNorm2.")

        new_state_dict[new_key] = value

    # 4. Load the remapped state dict
    model.load_state_dict(new_state_dict)
    print("successfully mapped the model weights to our architecture")

    return model

# Setup the draft and target models

## Load the draft model

In [41]:
GPT_CONFIG_124M_OPEN_AI = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": True,      # Query-key-value bias
    "weight_tying": False
}
GPT_CONFIG_DRAFT = GPT_CONFIG_124M_OPEN_AI

In [35]:
path_to_draft_model = "gpt2-small-124M.pth"

download_weights_from_openai(filename=path_to_draft_model)
draft_model = load_model(
    weights_path=path_to_draft_model,
    config=GPT_CONFIG_124M_OPEN_AI,
    device=pick_device(),
)

gpt2-small-124M.pth already exists.
successfully mapped the model weights to our architecture


## Load the Target Model

In [ ]:
_GPT_CONFIG_1558M_OPEN_AI = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # 
    "emb_dim": 1600,        # Embedding dimension
    "n_heads": 25,         # Number of attention heads
    "n_layers": 48,        # Number of transformer layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": True,      # Query-key-value bias
    "weight_tying": False
}

In [27]:
GPT_CONFIG_355M_OPEN_AI = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 1024, # 
    "emb_dim": 1024,        # Embedding dimension
    "n_heads": 16,         # Number of attention heads
    "n_layers": 24,        # Number of transformer layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": True,      # Query-key-value bias
    "weight_tying": False
}

In [34]:
path_to_target_model = "gpt2-medium-355M.pth"

download_weights_from_openai(filename=path_to_target_model)
target_model = load_model(
    weights_path=path_to_target_model,
    config=GPT_CONFIG_355M_OPEN_AI,
    device=pick_device(),
)

gpt2-medium-355M.pth already exists.
successfully mapped the model weights to our architecture


# Generate tokens with the draft model

## Warm-up: Sample the entire sequence from the draft model only

In [56]:
def generate(model, current_sequence, number_of_tokens, context_size):
    for _ in range(number_of_tokens):
        # clip the input to context_window size
        # ex: keep the last 5 tokens if the context size is 5
        cropped_context = current_sequence[:, -context_size:]
        # # convert the input into token ids
        # input_tokens = text_to_tokens(current_sequence)

        with torch.no_grad():
            logits, _kv_cache = model(cropped_context)

        # focus only on the last token's logits
        logits = logits[:, -1, :]

        # convert the logits into probabilities
        probas = torch.softmax(logits, dim=-1)  # batch, vocab_size

        # sample greedily - pick the one with the highest probability
        next_idx = torch.argmax(probas, dim=-1, keepdim=True)

        current_sequence = torch.cat(
            (current_sequence, next_idx), dim=1
        )  # batch, n_tokens+1

    return current_sequence

In [ ]:
# helper functions for tokenization

import tiktoken
from tokenization_helper import text_to_token_ids, token_ids_to_text

tokenizer = tiktoken.get_encoding("gpt2")

In [57]:
# call the draft model and verify it generates text

torch.manual_seed(123)

token_ids = generate(
    model=draft_model,
    current_sequence=text_to_token_ids("Every effort moves you", tokenizer).to(
        pick_device()
    ),
    number_of_tokens=15,
    context_size=GPT_CONFIG_DRAFT["context_length"],
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you can't make.


The only way to make a good game


## Generate K-tokens at a time using the draft model

In [59]:
# this is gonna be very similar to the generate function above,
# except that we'll also keep track the probability vector
# for the last token at each generation step


def draft_lookahead_tokens(model, current_sequence, k, context_size):
    # current_sequence is a 2D tensor: [1, sequence_length].
    draft_probs = []

    for _ in range(k):
        cropped_context = current_sequence[:, -context_size:]

        with torch.no_grad():
            # this tells PyTorch to disable the gradient calculation engine,
            # thereby saving memory (stops taking derivatives, so no training).
            # otherwise PyTorch remembers every operation (mul, add, etc.) so
            # that it can calc gradients during backprop much quicker.            
            logits, _kv_cache = model(cropped_context)

        # focus only on the last token
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)        
        logits = logits[:, -1, :]

        # The shape of probas will be [batch, vocab_size] 
        # (ex: [1, 50257]).
        probas = torch.softmax(logits, dim=-1)
        draft_probs.append(probas)
        # keep the dimension after performing the argmax
        # so, instead of returning a 1-D tensor of size [batch],
        # it returns 2-D tensor of size [batch, 1] - ex:
        # tensor([[4], [7], [1]]) # Shape is [3, 1], instead of
        # tensor([4, 7, 1])  # Shape is [3]
        next_idx = torch.argmax(probas, dim=-1, keepdim=True)

        # since we did keep dim above, we can cat the two
        # tensors since both are 2-D
        current_sequence = torch.cat(
            (current_sequence, next_idx), dim=1
        )  # batch, n_tokens+1

    return (
        current_sequence,
        # Stacks along a new dimension so we get a single
        # tensor of shape [K, 1, 50257]
        torch.stack(draft_probs, dim=0),
    )